# Training con embedding ChemBERTa 

Configurazione:
- nested cross-validation **5 outer × 5 inner**;
- stessa griglia di tuning del Gradient Boosting usata nel notebook originale;
- rappresentazioni considerate: **embedding completo** e **PCA 95%**;
- `raw` sempre incluso;
- `clean` opzionale tramite flag;
- PCA stimata **dentro la pipeline**, quindi senza leakage;
- allineamento esplicito tra embedding e target tramite `_id`;
- stessi outer split per tutte le rappresentazioni confrontate;
- salvataggio di risultati per fold, summary e predizioni out-of-fold.

La feature selection usata nella baseline a descrittori non viene applicata agli embedding ChemBERTa.

In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from pymongo import MongoClient

from sklearn.decomposition import PCA
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GridSearchCV, KFold
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

## Configurazione

In [ ]:
# Dataset e opzioni di caricamento
COLLECTION_NAME = "biceranoPolymers"
TARGET = "Tg"

RANDOM_STATE = 28
OUTER_FOLDS = 5
INNER_FOLDS = 5

# False = solo P-SMILES originale (run principale)
# True  = confronta anche gli embedding ottenuti rimuovendo [*]
COMPARE_CLEAN = False

PROJECT_NAME = "polymer_prediction"
ROOT = Path.cwd()

while ROOT != ROOT.parent and ROOT.name != PROJECT_NAME:
    ROOT = ROOT.parent

if ROOT.name != PROJECT_NAME:
    raise RuntimeError(f"Root del progetto {PROJECT_NAME!r} non trovata")

EMBEDDINGS_DIR = ROOT / "data" / "embeddings"
OUTPUT_DIR = ROOT / "results" / "chemberta"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VARIANTS = ["raw"] + (["clean"] if COMPARE_CLEAN else [])
REPRESENTATIONS = ["full", "pca95"]

# Usa la stessa griglia di tuning della baseline.
PARAM_GRID = {
    "regressor__n_estimators": [200, 400],
    "regressor__learning_rate": [0.05, 0.1],
    "regressor__max_depth": [2, 3],
    "regressor__subsample": [0.8, 1.0],
}

print("Dataset:", COLLECTION_NAME)
print("Varianti:", VARIANTS)
print("Rappresentazioni:", REPRESENTATIONS)
print(f"Nested CV: {OUTER_FOLDS} outer × {INNER_FOLDS} inner")
print("Embedding dir:", EMBEDDINGS_DIR.resolve())

## Caricamento target da MongoDB

In [ ]:
load_dotenv(ROOT / ".env")
MONGO_URI = os.getenv("MONGO_URI")

if not MONGO_URI:
    raise RuntimeError("MONGO_URI non trovato nel file .env")

client = MongoClient(MONGO_URI)
collection = client["PolymerPrediction"][COLLECTION_NAME]

# Ordina i campioni in modo riproducibile.
docs = list(collection.find({}).sort("_id", 1))
df = pd.DataFrame(docs)

if df.empty:
    raise RuntimeError(f"La collection {COLLECTION_NAME!r} è vuota")

if "properties" in df.columns:
    prop_df = pd.json_normalize(df["properties"])
    df = pd.concat([df.drop(columns=["properties"]), prop_df], axis=1)

df["_id"] = df["_id"].astype(str)
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

# Mantiene gli stessi criteri di pulizia del notebook originale quando SMILES è presente.
required = [TARGET]
if "smiles" in df.columns:
    required.append("smiles")

df = df.dropna(subset=required)

if "smiles" in df.columns:
    df = df[~df["smiles"].str.contains(r"\[\[", na=False)].copy()

df = df.set_index("_id")

if not df.index.is_unique:
    raise ValueError("_id duplicati nel dataset")

y_all = df[TARGET].copy()

print("Target disponibili:", len(y_all))
print(f"σ({TARGET}) = {y_all.std(ddof=1):.4f}")

## Caricamento e allineamento embedding

In [ ]:
def load_embeddings(variant):
    path = EMBEDDINGS_DIR / f"chemberta_{COLLECTION_NAME}_{variant}.csv"

    if not path.exists():
        raise FileNotFoundError(f"Embedding non trovati: {path.resolve()}")

    X = pd.read_csv(path)

    if "_id" not in X.columns:
        raise ValueError(f"{path} non contiene la colonna _id")

    X["_id"] = X["_id"].astype(str)
    X = X.set_index("_id")

    if not X.index.is_unique:
        raise ValueError(f"_id duplicati in {path}")

    if X.isna().any().any():
        raise ValueError(f"NaN trovati in {path}")

    common_ids = y_all.index.intersection(X.index, sort=False)

    X = X.loc[common_ids]
    y = y_all.loc[common_ids]

    assert X.index.equals(y.index)

    return X, y


data = {variant: load_embeddings(variant) for variant in VARIANTS}

# Se sono attive entrambe le varianti, usa gli stessi campioni per il confronto.
if len(data) > 1:
    shared_ids = y_all.index

    for X, _ in data.values():
        shared_ids = shared_ids[shared_ids.isin(X.index)]

    data = {
        variant: (X.loc[shared_ids], y_all.loc[shared_ids])
        for variant, (X, _) in data.items()
    }

for variant, (X, y) in data.items():
    print(f"{variant:>5}: X={X.shape}, y={y.shape}")

# Tutte le varianti devono usare gli stessi campioni per un confronto appaiato.
reference_ids = next(iter(data.values()))[0].index
for X, _ in data.values():
    assert X.index.equals(reference_ids)

## Nested cross-validation

Gli outer split vengono creati una sola volta e riutilizzati per `full`, `pca95` e, se attivo, `clean`.

Per `pca95`, `PCA(n_components=0.95)` è parte della `Pipeline`. Di conseguenza viene rifittata in ogni split della cross-validation interna e poi sul training outer completo: il test fold esterno non contribuisce mai al calcolo delle componenti.

In [ ]:
X_reference, y_reference = next(iter(data.values()))

outer_cv = KFold(
    n_splits=OUTER_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

# Fissa gli split cosi tutte le configurazioni vengono confrontate sugli stessi dati.
outer_splits = list(outer_cv.split(X_reference, y_reference))

fold_rows = []
oof_rows = []

for variant, (X, y) in data.items():
    sigma = y.std(ddof=1)

    for representation in REPRESENTATIONS:
        print("\n" + "=" * 70)
        print(f"{COLLECTION_NAME} | {variant} | {representation}")
        print("=" * 70)

        for fold, (train_idx, test_idx) in enumerate(outer_splits, start=1):
            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y.iloc[train_idx]
            y_test = y.iloc[test_idx]

            if representation == "full":
                reducer = "passthrough"
            else:
                reducer = PCA(
                    n_components=0.95,
                    svd_solver="full",
                )

            pipeline = Pipeline([
                ("reduce", reducer),
                ("regressor", GradientBoostingRegressor(
                    random_state=RANDOM_STATE
                )),
            ])

            inner_cv = KFold(
                n_splits=INNER_FOLDS,
                shuffle=True,
                random_state=RANDOM_STATE,
            )

            search = GridSearchCV(
                estimator=pipeline,
                param_grid=PARAM_GRID,
                scoring="neg_mean_absolute_error",
                cv=inner_cv,
                n_jobs=-1,
                refit=True,
                verbose=1,
            )

            search.fit(X_train, y_train)

            best_model = search.best_estimator_
            pred = best_model.predict(X_test)

            mae = mean_absolute_error(y_test, pred)
            r2 = r2_score(y_test, pred)

            if representation == "pca95":
                n_components = best_model.named_steps["reduce"].n_components_
            else:
                n_components = X.shape[1]

            fold_rows.append({
                "dataset": COLLECTION_NAME,
                "variant": variant,
                "representation": representation,
                "fold": fold,
                "n_train": len(train_idx),
                "n_test": len(test_idx),
                "n_components": int(n_components),
                "mae": mae,
                "r2": r2,
                "mae_over_sigma": mae / sigma,
                "best_n_estimators": search.best_params_["regressor__n_estimators"],
                "best_learning_rate": search.best_params_["regressor__learning_rate"],
                "best_max_depth": search.best_params_["regressor__max_depth"],
                "best_subsample": search.best_params_["regressor__subsample"],
            })

            for sample_id, true, prediction in zip(
                X_test.index, y_test.to_numpy(), pred
            ):
                oof_rows.append({
                    "_id": sample_id,
                    "dataset": COLLECTION_NAME,
                    "variant": variant,
                    "representation": representation,
                    "fold": fold,
                    "y_true": true,
                    "y_pred": prediction,
                    "error": prediction - true,
                    "abs_error": abs(prediction - true),
                })

            print(
                f"Fold {fold}: "
                f"MAE={mae:.2f} K | "
                f"R²={r2:.3f} | "
                f"MAE/σ={mae/sigma:.3f} | "
                f"components={n_components}"
            )
            print("  best:", search.best_params_)

## Risultati

In [ ]:
fold_results = pd.DataFrame(fold_rows)
oof_results = pd.DataFrame(oof_rows)

summary = (
    fold_results
    .groupby(["dataset", "variant", "representation"], as_index=False)
    .agg(
        n=("n_test", "sum"),
        n_components_mean=("n_components", "mean"),
        n_components_std=("n_components", "std"),
        mae_mean=("mae", "mean"),
        mae_std=("mae", "std"),
        r2_mean=("r2", "mean"),
        r2_std=("r2", "std"),
        mae_over_sigma_mean=("mae_over_sigma", "mean"),
        mae_over_sigma_std=("mae_over_sigma", "std"),
    )
    .sort_values(["variant", "mae_mean"])
    .reset_index(drop=True)
)

summary

## Salvataggio

In [ ]:
tag = "raw_clean" if COMPARE_CLEAN else "raw"

fold_path = OUTPUT_DIR / f"chemberta_{COLLECTION_NAME}_{tag}_nestedcv_folds.csv"
summary_path = OUTPUT_DIR / f"chemberta_{COLLECTION_NAME}_{tag}_nestedcv_summary.csv"
oof_path = OUTPUT_DIR / f"chemberta_{COLLECTION_NAME}_{tag}_nestedcv_oof.csv"

fold_results.to_csv(fold_path, index=False)
summary.to_csv(summary_path, index=False)
oof_results.to_csv(oof_path, index=False)

print("Salvati:")
print(" -", fold_path)
print(" -", summary_path)
print(" -", oof_path)